## Setup
**What we use:** pandas/NumPy for data handling, Matplotlib/Seaborn for plots, scikit-learn for pipelines, grouped validation, tuning and metrics, joblib for saving models.

# Parkinson's Symptom Severity Prediction — Advanced ML Project

**Dataset:** `parkinsons_updrs(1).csv`  
**Targets:** `motor_UPDRS` and `total_UPDRS`  
**Task:** Supervised regression, evaluated on patients held out from training.

## How to use
1. Run cells from top to bottom.
2. When prompted, upload `parkinsons_updrs(1).csv` if it is not already in the notebook directory.
3. Model search may take several minutes.
4. Run the dashboard after exporting the model artifacts.

**Clinical caution:** This is an educational/research prototype. It does not diagnose Parkinson's disease or replace a clinician. UPDRS scores are estimated scores, not disease diagnoses.

In [ ]:
%pip -q install pandas numpy matplotlib seaborn scikit-learn joblib streamlit

In [ ]:
import os, json, warnings, joblib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")
PROJECT_DIR = Path(".") if Path.cwd().name == "Parkinsons_Severity_ML" else Path("Parkinsons_Severity_ML")
for folder in ["data/raw", "data/processed", "models", "reports/figures"]:
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)

print("Environment ready.")

# MODULE 1 — Dataset Management
**What we use:** schema checks, dataset audit, patient-level summaries and raw-data backup. This confirms the data structure before analysis.

In [ ]:
# If the CSV is already in the current directory, this loads it.
# Otherwise, in Google Colab, it will ask you to upload the file.
csv_candidates = [
    Path("parkinsons_updrs(1).csv"),
    Path("parkinsons_updrs.csv"),
    Path("/mnt/data/parkinsons_updrs(1).csv")
]
csv_path = next((p for p in csv_candidates if p.exists()), None)

if csv_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_path = Path(next(iter(uploaded)))
    except Exception as e:
        raise FileNotFoundError(
            "Place parkinsons_updrs(1).csv in the notebook directory or upload it."
        ) from e

df_raw = pd.read_csv(csv_path)
print("Loaded:", csv_path)
print("Shape:", df_raw.shape)
display(df_raw.head())

In [ ]:
required_columns = [
    "subject#", "age", "sex", "test_time",
    "motor_UPDRS", "total_UPDRS",
    "Jitter(%)", "Jitter(Abs)", "Jitter:RAP", "Jitter:PPQ5", "Jitter:DDP",
    "Shimmer", "Shimmer(dB)", "Shimmer:APQ3", "Shimmer:APQ5",
    "Shimmer:APQ11", "Shimmer:DDA", "NHR", "HNR", "RPDE", "DFA", "PPE"
]
missing_schema = sorted(set(required_columns) - set(df_raw.columns))
if missing_schema:
    raise ValueError(f"Dataset is missing required columns: {missing_schema}")

audit = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing_count": df_raw.isna().sum(),
    "missing_percent": (100 * df_raw.isna().mean()).round(2),
    "unique_values": df_raw.nunique()
})
display(audit)
print("Exact duplicate rows:", df_raw.duplicated().sum())
print("Unique patients:", df_raw["subject#"].nunique())
df_raw.to_csv(PROJECT_DIR / "data/raw/parkinsons_updrs.csv", index=False)
audit.to_csv(PROJECT_DIR / "reports/dataset_audit.csv")

In [ ]:
patient_summary = (
    df_raw.groupby("subject#")
    .agg(
        records=("subject#", "size"),
        age=("age", "first"),
        sex=("sex", "first"),
        motor_mean=("motor_UPDRS", "mean"),
        total_mean=("total_UPDRS", "mean"),
        first_test_time=("test_time", "min"),
        last_test_time=("test_time", "max")
    )
    .reset_index()
)
display(patient_summary.head())
print("Patients with repeated records:", int((patient_summary["records"] > 1).sum()))
patient_summary.to_csv(PROJECT_DIR / "reports/patient_summary.csv", index=False)

# MODULE 2 — Advanced Preprocessing
**What we use:** duplicate removal, infinity handling, target/input separation and a grouped train/test split. Keeping all records from a patient in one split reduces patient leakage.

In [ ]:
df = df_raw.copy()
df = df.drop_duplicates().reset_index(drop=True)

numeric_cols = df.select_dtypes(include=np.number).columns
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)

# Target rows cannot be used for supervised training if their target is missing.
targets = ["motor_UPDRS", "total_UPDRS"]
df = df.dropna(subset=targets).reset_index(drop=True)

# Do not use patient ID or either UPDRS target as predictors.
excluded = ["subject#", "motor_UPDRS", "total_UPDRS"]
base_feature_columns = [c for c in df.columns if c not in excluded]

X_base = df[base_feature_columns].copy()
y = df[targets].copy()
groups = df["subject#"].copy()

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X_base, y, groups=groups))

X_train_base = X_base.iloc[train_idx].copy()
X_test_base = X_base.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()
groups_train = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

print("Training rows:", len(X_train_base), "| Test rows:", len(X_test_base))
print("Training patients:", groups_train.nunique(), "| Test patients:", groups_test.nunique())
print("Patients shared between train and test:",
      len(set(groups_train.unique()) & set(groups_test.unique())))
assert len(set(groups_train.unique()) & set(groups_test.unique())) == 0
print("Preprocessing split ready.")

# MODULE 3 — Statistical EDA
**What we use:** distributions, correlations, patient-level trends and basic group summaries to understand the dataset before modeling. EDA is descriptive; correlation is not causation.

In [ ]:
display(df.describe(include="all").T)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df["motor_UPDRS"], kde=True, ax=axes[0])
axes[0].set_title("Motor UPDRS distribution")
sns.histplot(df["total_UPDRS"], kde=True, ax=axes[1])
axes[1].set_title("Total UPDRS distribution")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 10))
corr = df.drop(columns=["subject#"]).corr(numeric_only=True)
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation heatmap")
plt.tight_layout()
plt.show()

In [ ]:
# Average motor UPDRS over time, using patient-level means at each test time.
# This is descriptive only; it is not a causal progression estimate.
time_trend = (
    df.groupby("test_time", as_index=False)
      .agg(motor_mean=("motor_UPDRS", "mean"),
           total_mean=("total_UPDRS", "mean"))
      .sort_values("test_time")
)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(time_trend["test_time"], time_trend["motor_mean"], label="Motor UPDRS")
ax.plot(time_trend["test_time"], time_trend["total_mean"], label="Total UPDRS")
ax.set_xlabel("Test time")
ax.set_ylabel("Mean score")
ax.set_title("Mean UPDRS by test time")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Correlations of features with each target
display(
    df.drop(columns=["subject#"])
      .corr(numeric_only=True)[targets]
      .sort_values("motor_UPDRS", ascending=False)
)

# MODULE 4 — Advanced Feature Engineering
**What we use:** domain-inspired combinations and nonlinear terms from input columns only. Target scores and patient IDs are never used to create predictors. Keep engineered features only if grouped validation shows they help.

In [ ]:
def engineer_features(data: pd.DataFrame) -> pd.DataFrame:
    out = data.copy()

    # Ensure expected input columns exist
    expected = [c for c in base_feature_columns if c not in out.columns]
    if expected:
        raise ValueError(f"Missing input features: {expected}")

    # Numeric age band; NaN values are later handled by the pipeline
    out["age_band"] = pd.cut(
        out["age"], bins=[0, 40, 50, 60, 70, 120],
        labels=False, include_lowest=True
    ).astype(float)

    jitter_cols = ["Jitter(%)", "Jitter(Abs)", "Jitter:RAP", "Jitter:PPQ5", "Jitter:DDP"]
    shimmer_cols = ["Shimmer", "Shimmer(dB)", "Shimmer:APQ3", "Shimmer:APQ5",
                    "Shimmer:APQ11", "Shimmer:DDA"]

    out["jitter_mean"] = out[jitter_cols].mean(axis=1)
    out["shimmer_mean"] = out[shimmer_cols].mean(axis=1)
    out["voice_noise_ratio"] = out["NHR"] / (out["HNR"].abs() + 1e-6)
    out["PPE_squared"] = out["PPE"] ** 2
    out["RPDE_squared"] = out["RPDE"] ** 2
    out["DFA_x_PPE"] = out["DFA"] * out["PPE"]

    return out.replace([np.inf, -np.inf], np.nan)

X_train = engineer_features(X_train_base)
X_test = engineer_features(X_test_base)
feature_columns = X_train.columns.tolist()

print("Base feature count:", len(base_feature_columns))
print("Engineered feature count:", len(feature_columns))
display(X_train.head())

In [ ]:
X_train.to_csv(PROJECT_DIR / "data/processed/X_train_features.csv", index=False)
X_test.to_csv(PROJECT_DIR / "data/processed/X_test_features.csv", index=False)
y_train.to_csv(PROJECT_DIR / "data/processed/y_train.csv", index=False)
y_test.to_csv(PROJECT_DIR / "data/processed/y_test.csv", index=False)
print("Engineered datasets saved.")

# MODULE 5 — Model Training & Optimization
**What we use:** Ridge, Extra Trees and HistGradientBoosting regressors; median imputation; scaling for consistent pipelines; RandomizedSearchCV with GroupKFold and patient groups. Hyperparameter search is run on training patients only.

In [ ]:
# Model pipelines; tree models do not require scaling, but it does not invalidate them.
# Ridge benefits from standardization.
from sklearn.base import clone

model_spaces = [
    {
        "model": [Ridge()],
        "model__alpha": [0.1, 1.0, 10.0, 50.0, 100.0]
    },
    {
        "model": [ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=-1)],
        "model__n_estimators": [100, 200, 300],
        "model__max_depth": [None, 10, 20, 30],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": [0.7, 1.0]
    },
    {
        "model": [HistGradientBoostingRegressor(random_state=RANDOM_STATE)],
        "model__learning_rate": [0.03, 0.05, 0.1],
        "model__max_iter": [100, 200],
        "model__max_leaf_nodes": [15, 31],
        "model__l2_regularization": [0.0, 0.1, 1.0]
    }
]

base_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge())
])

n_splits = min(3, groups_train.nunique())
if n_splits < 2:
    raise ValueError("At least two training patients are required for GroupKFold.")

cv = GroupKFold(n_splits=n_splits)
best_models = {}
search_summaries = []

for target in targets:
    print(f"\nTuning target: {target}")
    search = RandomizedSearchCV(
        estimator=base_pipeline,
        param_distributions=model_spaces,
        n_iter=8,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        refit=True,
        verbose=1,
        error_score="raise"
    )
    search.fit(X_train, y_train[target], groups=groups_train)

    best_models[target] = search.best_estimator_
    search_summaries.append({
        "target": target,
        "best_cv_RMSE": -search.best_score_,
        "best_model": type(search.best_estimator_.named_steps["model"]).__name__,
        "best_params": str(search.best_params_)
    })
    print("Best grouped CV RMSE:", round(-search.best_score_, 4))
    print("Best estimator:", type(search.best_estimator_.named_steps["model"]).__name__)
    print("Best parameters:", search.best_params_)

search_summary_df = pd.DataFrame(search_summaries)
display(search_summary_df)
search_summary_df.to_csv(PROJECT_DIR / "reports/hyperparameter_search.csv", index=False)

# MODULE 6 — Patient-Level Model Evaluation
**What we use:** untouched patient-level test data and MAE, RMSE, R², residual and actual-vs-predicted plots. The test set is used for final evaluation, not hyperparameter tuning.

In [ ]:
evaluation_rows = []
test_predictions = {}

for target, model in best_models.items():
    pred = model.predict(X_test)
    test_predictions[target] = pred
    evaluation_rows.append({
        "Target": target,
        "Model": type(model.named_steps["model"]).__name__,
        "MAE": mean_absolute_error(y_test[target], pred),
        "RMSE": np.sqrt(mean_squared_error(y_test[target], pred)),
        "R2": r2_score(y_test[target], pred)
    })

evaluation_df = pd.DataFrame(evaluation_rows)
display(evaluation_df)
evaluation_df.to_csv(PROJECT_DIR / "reports/model_metrics.csv", index=False)

In [ ]:
for target in targets:
    pred = test_predictions[target]
    actual = y_test[target].to_numpy()
    residual = actual - pred

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].scatter(actual, pred, alpha=0.45)
    lo = min(actual.min(), pred.min())
    hi = max(actual.max(), pred.max())
    axes[0].plot([lo, hi], [lo, hi], "--")
    axes[0].set_xlabel("Actual")
    axes[0].set_ylabel("Predicted")
    axes[0].set_title(f"Actual vs predicted: {target}")

    axes[1].scatter(pred, residual, alpha=0.45)
    axes[1].axhline(0, linestyle="--")
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("Residual (actual - predicted)")
    axes[1].set_title(f"Residuals: {target}")

    plt.tight_layout()
    plt.show()

In [ ]:
# Patient-level summary of test performance (not treating rows as independent patients)
test_eval = X_test.copy()
test_eval["subject#"] = groups_test.to_numpy()
for target in targets:
    test_eval[f"actual_{target}"] = y_test[target].to_numpy()
    test_eval[f"pred_{target}"] = test_predictions[target]

patient_errors = []
for patient, part in test_eval.groupby("subject#"):
    row = {"subject#": patient, "records": len(part)}
    for target in targets:
        row[f"MAE_{target}"] = mean_absolute_error(
            part[f"actual_{target}"], part[f"pred_{target}"]
        )
    patient_errors.append(row)

patient_error_df = pd.DataFrame(patient_errors)
display(patient_error_df.describe().T)
patient_error_df.to_csv(PROJECT_DIR / "reports/patient_level_errors.csv", index=False)

# MODULE 7 — Explainable Severity Prediction
**What we use:** reusable prediction function, feature-aligned input validation and a sample prediction. The output is an estimated UPDRS score, not a diagnosis or a validated severity category.

In [ ]:
def predict_updrs(input_record):
    # Accept a dict or a one-row DataFrame of raw input features.
    row = pd.DataFrame([input_record]) if isinstance(input_record, dict) else input_record.copy()

    missing = [c for c in base_feature_columns if c not in row.columns]
    if missing:
        raise ValueError(f"Missing required features: {missing}")

    row = row[base_feature_columns]
    row_fe = engineer_features(row)
    row_fe = row_fe[feature_columns]

    output = {}
    for target, model in best_models.items():
        output[target] = float(model.predict(row_fe)[0])
    return output

sample_record = X_test_base.iloc[0].to_dict()
sample_prediction = predict_updrs(sample_record)
print("Predictions for one held-out record:")
print(json.dumps(sample_prediction, indent=2))
print("Actual motor_UPDRS:", float(y_test.iloc[0]["motor_UPDRS"]))
print("Actual total_UPDRS:", float(y_test.iloc[0]["total_UPDRS"]))

In [ ]:
# Export a reusable sample input template for the dashboard/API.
pd.DataFrame([sample_record]).to_csv(
    PROJECT_DIR / "reports/sample_input_template.csv", index=False
)
print("Sample input template saved.")

# MODULE 8 — Interactive ML Dashboard
**What we use:** Streamlit for a local web app with dataset overview, model metrics, EDA and predictions. Run the app after the notebook has saved model artifacts.

In [ ]:
# Save models and metadata for the Streamlit app.
for target, model in best_models.items():
    joblib.dump(model, PROJECT_DIR / f"models/{target}_model.joblib")

metadata = {
    "targets": targets,
    "base_feature_columns": base_feature_columns,
    "feature_columns": feature_columns,
    "random_state": RANDOM_STATE
}
with open(PROJECT_DIR / "models/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Model artifacts exported.")
print("Dashboard file will be created in dashboard/app.py.")

## Dashboard application file
The next cell writes `dashboard/app.py`. It uses the exact feature engineering used during training.

In [ ]:
app_code = r"""
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import streamlit as st
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path(__file__).resolve().parents[1]
DATA_PATH = ROOT / "data/raw/parkinsons_updrs.csv"
METRICS_PATH = ROOT / "reports/model_metrics.csv"
META_PATH = ROOT / "models/metadata.json"

st.set_page_config(page_title="Parkinson's Severity ML", layout="wide")
st.title("Parkinson's Symptom Severity Prediction")
st.caption("Educational research prototype — not a clinical diagnostic system.")

@st.cache_data
def load_data():
    return pd.read_csv(DATA_PATH)

@st.cache_data
def load_metrics():
    return pd.read_csv(METRICS_PATH)

@st.cache_data
def load_metadata():
    with open(META_PATH) as f:
        return json.load(f)

@st.cache_resource
def load_model(target):
    return joblib.load(ROOT / "models" / f"{target}_model.joblib")

def engineer_features(data):
    out = data.copy()
    out["age_band"] = pd.cut(
        out["age"], bins=[0, 40, 50, 60, 70, 120],
        labels=False, include_lowest=True
    ).astype(float)
    jitter_cols = ["Jitter(%)", "Jitter(Abs)", "Jitter:RAP", "Jitter:PPQ5", "Jitter:DDP"]
    shimmer_cols = ["Shimmer", "Shimmer(dB)", "Shimmer:APQ3", "Shimmer:APQ5",
                    "Shimmer:APQ11", "Shimmer:DDA"]
    out["jitter_mean"] = out[jitter_cols].mean(axis=1)
    out["shimmer_mean"] = out[shimmer_cols].mean(axis=1)
    out["voice_noise_ratio"] = out["NHR"] / (out["HNR"].abs() + 1e-6)
    out["PPE_squared"] = out["PPE"] ** 2
    out["RPDE_squared"] = out["RPDE"] ** 2
    out["DFA_x_PPE"] = out["DFA"] * out["PPE"]
    return out.replace([np.inf, -np.inf], np.nan)

df = load_data()
metrics = load_metrics()
meta = load_metadata()

page = st.sidebar.radio(
    "Navigation",
    ["Overview", "Dataset Explorer", "EDA", "Model Evaluation", "Predict UPDRS"]
)

if page == "Overview":
    c1, c2, c3 = st.columns(3)
    c1.metric("Records", f"{len(df):,}")
    c2.metric("Patients", f"{df['subject#'].nunique():,}")
    c3.metric("Columns", len(df.columns))
    st.subheader("Project")
    st.write("Patient-level regression models estimate motor and total UPDRS scores.")
    st.warning("Do not use this prototype for diagnosis, treatment, or clinical decisions.")

elif page == "Dataset Explorer":
    st.subheader("Dataset preview")
    st.dataframe(df.head(200), use_container_width=True)
    st.subheader("Descriptive statistics")
    st.dataframe(df.describe().T, use_container_width=True)
    st.subheader("Missing values")
    st.dataframe(df.isna().sum().rename("Missing count"))

elif page == "EDA":
    target = st.selectbox("Target distribution", ["motor_UPDRS", "total_UPDRS"])
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(df[target], kde=True, ax=ax)
    ax.set_title(f"Distribution of {target}")
    st.pyplot(fig)
    fig, ax = plt.subplots(figsize=(11, 7))
    sns.heatmap(df.drop(columns=["subject#"]).corr(numeric_only=True),
                cmap="coolwarm", center=0, ax=ax)
    st.pyplot(fig)

elif page == "Model Evaluation":
    st.subheader("Held-out patient test results")
    st.dataframe(metrics, use_container_width=True)
    metric_name = st.selectbox("Metric", ["MAE", "RMSE", "R2"])
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(metrics["Target"], metrics[metric_name])
    ax.set_ylabel(metric_name)
    ax.set_title(f"{metric_name} by target")
    st.pyplot(fig)

elif page == "Predict UPDRS":
    st.write("Enter measured inputs. Defaults are illustrative only; use real measured values only in approved research settings.")
    template = pd.read_csv(ROOT / "reports/sample_input_template.csv").iloc[0].to_dict()
    with st.form("predict_form"):
        vals = {}
        cols = st.columns(3)
        for i, name in enumerate(meta["base_feature_columns"]):
            val = float(template[name])
            with cols[i % 3]:
                if name == "sex":
                    vals[name] = st.selectbox(name, [0, 1], index=int(val) if val in [0, 1] else 0)
                elif name == "age":
                    vals[name] = st.number_input(name, min_value=18.0, max_value=100.0, value=float(np.clip(val, 18, 100)))
                else:
                    vals[name] = st.number_input(name, value=val, format="%.6f")
        submitted = st.form_submit_button("Estimate UPDRS scores")
    if submitted:
        row = pd.DataFrame([vals])[meta["base_feature_columns"]]
        row = engineer_features(row)[meta["feature_columns"]]
        a, b = st.columns(2)
        for col, target in [(a, "motor_UPDRS"), (b, "total_UPDRS")]:
            pred = float(load_model(target).predict(row)[0])
            col.metric(f"Estimated {target}", f"{pred:.2f}")
        st.info("These are model estimates, not a diagnosis or validated clinical severity category.")
"""
app_path = PROJECT_DIR / "dashboard/app.py"
app_path.write_text(app_code)
print("Created:", app_path)

# MODULE 9 — Model Interpretability
**What we use:** permutation importance and partial dependence to examine how features affect predictions. Permutation importance is model-agnostic; correlated features can share or mask importance. SHAP is optional and can be computationally expensive.

In [ ]:
# Permutation importance is computed on a capped test sample to reduce runtime.
interpretation_rows = []
sample_n = min(500, len(X_test))
X_interpret = X_test.sample(sample_n, random_state=RANDOM_STATE)
y_interpret = y_test.loc[X_interpret.index]

for target, model in best_models.items():
    perm = permutation_importance(
        model,
        X_interpret,
        y_interpret[target],
        scoring="neg_mean_absolute_error",
        n_repeats=5,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    imp = pd.DataFrame({
        "feature": feature_columns,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
        "target": target
    }).sort_values("importance_mean", ascending=False)

    interpretation_rows.append(imp)

    display(imp.head(15))

    top = imp.head(15).sort_values("importance_mean")
    plt.figure(figsize=(8, 6))
    plt.barh(top["feature"], top["importance_mean"], xerr=top["importance_std"])
    plt.title(f"Permutation importance — {target}")
    plt.xlabel("Decrease in negative MAE when shuffled (higher is more important)")
    plt.tight_layout()
    plt.show()

importance_df = pd.concat(interpretation_rows, ignore_index=True)
importance_df.to_csv(PROJECT_DIR / "reports/permutation_importance.csv", index=False)

In [ ]:
# Optional SHAP analysis. If the installed SHAP version/model combination fails,
# permutation importance above remains the primary explanation method.
try:
    import shap
    target = "motor_UPDRS"
    model = best_models[target]
    estimator = model.named_steps["model"]
    X_shap = X_interpret.sample(min(100, len(X_interpret)), random_state=RANDOM_STATE)

    # For tree estimators, explain the estimator on imputed/scaled features.
    X_transformed = model.named_steps["scaler"].transform(
        model.named_steps["imputer"].transform(X_shap)
    )
    explainer = shap.Explainer(estimator, X_transformed)
    shap_values = explainer(X_transformed)
    shap.plots.beeswarm(shap_values, max_display=15)
except Exception as exc:
    print("Optional SHAP plot skipped:", repr(exc))

# MODULE 10 — Deployment & Monitoring
**What we use:** versioned model artifacts, input schema validation, prediction logging and a basic feature-drift check. Drift alerts are investigation prompts, not proof that model accuracy has fallen.

In [ ]:
# Save final artifacts and a lightweight reference profile from training data.
for target, model in best_models.items():
    joblib.dump(model, PROJECT_DIR / f"models/{target}_model.joblib")

metadata = {
    "targets": targets,
    "base_feature_columns": base_feature_columns,
    "feature_columns": feature_columns,
    "random_state": RANDOM_STATE,
    "project_version": "1.0"
}
with open(PROJECT_DIR / "models/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

reference_profile = X_train[base_feature_columns].describe().T
reference_profile.to_csv(PROJECT_DIR / "reports/reference_profile.csv")

print("Model artifacts and reference profile saved.")

In [ ]:
def validate_prediction_input(record, expected_columns=base_feature_columns):
    row = pd.DataFrame([record]) if isinstance(record, dict) else record.copy()
    missing = [c for c in expected_columns if c not in row.columns]
    extra = [c for c in row.columns if c not in expected_columns]
    if missing:
        raise ValueError(f"Missing required features: {missing}")
    if extra:
        print("Ignoring extra columns:", extra)
    row = row[expected_columns]
    for col in row.columns:
        row[col] = pd.to_numeric(row[col], errors="coerce")
    if row.isna().all(axis=1).iloc[0]:
        raise ValueError("All input values are missing or non-numeric.")
    return row

def monitor_feature_drift(reference_df, new_df, columns, threshold=0.25):
    # Simple standardized mean shift for numeric features; heuristic, not a formal test.
    rows = []
    for col in columns:
        ref = pd.to_numeric(reference_df[col], errors="coerce").dropna()
        new = pd.to_numeric(new_df[col], errors="coerce").dropna()
        if len(ref) == 0 or len(new) == 0:
            shift = np.nan
        else:
            scale = ref.std(ddof=0)
            shift = abs(new.mean() - ref.mean()) / (scale if scale > 1e-12 else 1.0)
        rows.append({
            "feature": col,
            "standardized_mean_shift": shift,
            "flag_for_review": bool(pd.notna(shift) and shift > threshold)
        })
    return pd.DataFrame(rows).sort_values(
        "standardized_mean_shift", ascending=False, na_position="last"
    )

# Demonstrate schema validation and a small drift report using held-out inputs.
validated_sample = validate_prediction_input(sample_record)
print("Input validation passed:", validated_sample.shape)

drift_report = monitor_feature_drift(
    X_train_base,
    X_test_base,
    base_feature_columns
)
display(drift_report.head(15))
drift_report.to_csv(PROJECT_DIR / "reports/feature_drift_report.csv", index=False)

In [ ]:
# Save prediction log example (avoid storing identifiable patient data in real deployments).
prediction = predict_updrs(sample_record)
prediction_log = pd.DataFrame([{
    "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
    "motor_UPDRS_estimate": prediction["motor_UPDRS"],
    "total_UPDRS_estimate": prediction["total_UPDRS"],
    "model_version": "1.0"
}])
prediction_log.to_csv(
    PROJECT_DIR / "reports/prediction_log_example.csv",
    index=False
)

print("All 10 modules completed.")
print("Project directory:", PROJECT_DIR.resolve())
print("Dashboard command: streamlit run Parkinsons_Severity_ML/dashboard/app.py")